# P105 — Product Cannibalization & Demand Shift Analysis
## Phase 2: Data Cleaning & Feature Engineering

**Objective:** Clean the raw data and create new features needed for cannibalization analysis.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

LAUNCH_DATE = pd.Timestamp('2024-06-01')
LAUNCHED_PRODUCTS = ['P1', 'P2', 'P3', 'P4', 'P5']
AFFECTED_GROUPS = ['G1', 'G2']

## Load Raw Data

In [ ]:
df = pd.read_csv("../Data/advanced_cannibalization_dataset.csv")
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

---
## Part A: Data Cleaning

### A1. Fix Data Types

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Stock_Available'] = df['Stock_Available'].astype(int)
df['Launch_Flag'] = df['Launch_Flag'].astype(int)

print("Date            -> datetime64")
print("Stock_Available -> int (0/1)")
print("Launch_Flag     -> int (0/1)")

### A2. Validate Revenue = Price × Sales

In [ ]:
revenue_check = (df['Price'] * df['Sales']).round(2)
mismatch = (df['Revenue'].round(2) != revenue_check).sum()

print(f"Revenue mismatches: {mismatch}")
if mismatch > 0:
    print("Recalculating revenue...")
    df['Revenue'] = (df['Price'] * df['Sales']).round(2)
else:
    print("All revenue values are consistent.")

### A3. Validate Rating Range (1–5)

In [ ]:
invalid = df[(df['Rating'] < 1) | (df['Rating'] > 5)].shape[0]
print(f"Out-of-range ratings: {invalid}")

neg_price   = (df['Price'] <= 0).sum()
neg_sales   = (df['Sales'] <= 0).sum()
print(f"Negative Price : {neg_price}")
print(f"Negative Sales : {neg_sales}")

### A4. Standardize & Sort

In [ ]:
# Strip whitespace from text columns
for col in ['Product_ID', 'Category', 'Product_Group', 'Region']:
    df[col] = df[col].str.strip()

# Sort by Product and Date
df.sort_values(['Product_ID', 'Date'], inplace=True)
df.reset_index(drop=True, inplace=True)

print("Text columns cleaned and data sorted.")
print("Part A Complete — Data is clean.")

---
## Part B: Feature Engineering

### B1. Before / After Launch Period

In [ ]:
df['Period_Flag'] = np.where(df['Date'] < LAUNCH_DATE, 'Before_Launch', 'After_Launch')

print(df['Period_Flag'].value_counts().to_string())

### B2. Tag Launched & Affected Products

In [ ]:
# Is this a newly launched product?
df['Is_Launched_Product'] = df['Product_ID'].isin(LAUNCHED_PRODUCTS).astype(int)

# Is this product in a group affected by new launches?
df['Is_Affected_Group'] = df['Product_Group'].isin(AFFECTED_GROUPS).astype(int)

print(f"Launched product records : {df['Is_Launched_Product'].sum():,}")
print(f"Affected group records  : {df['Is_Affected_Group'].sum():,}")

### B3. Time Features

In [ ]:
df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['Quarter']    = df['Date'].dt.quarter
df['Year_Month'] = df['Date'].dt.to_period('M').astype(str)

print("Added: Year, Month, Month_Name, Quarter, Year_Month")

### B4. Revenue Contribution (%)

In [ ]:
period_total = df.groupby('Period_Flag')['Revenue'].transform('sum')
df['Revenue_Contribution_Pct'] = (df['Revenue'] / period_total * 100).round(4)

print("Revenue_Contribution_Pct created.")

### B5. Sales Change (Before vs After) per Product

In [ ]:
sales_before = df[df['Period_Flag'] == 'Before_Launch'].groupby('Product_ID')['Sales'].mean()
sales_after  = df[df['Period_Flag'] == 'After_Launch'].groupby('Product_ID')['Sales'].mean()

sales_comp = pd.DataFrame({
    'Avg_Before': sales_before.round(2),
    'Avg_After': sales_after.round(2)
})
sales_comp['Change_Pct'] = ((sales_comp['Avg_After'] - sales_comp['Avg_Before']) / sales_comp['Avg_Before'] * 100).round(2)

print(sales_comp.to_string())

# Merge back
df = df.merge(sales_comp[['Change_Pct']].rename(columns={'Change_Pct': 'Sales_Change_Pct'}),
              on='Product_ID', how='left')

### B6. Cannibalization Flag

In [ ]:
# A product is cannibalized if:
# - Not a launched product
# - In an affected group (G1 or G2)
# - Sales declined > 5% after launch

df['Cannibalization_Flag'] = (
    (df['Is_Launched_Product'] == 0) &
    (df['Is_Affected_Group'] == 1) &
    (df['Sales_Change_Pct'] < -5)
).astype(int)

cann_products = df[df['Cannibalization_Flag'] == 1]['Product_ID'].unique()
print(f"Cannibalized products: {sorted(cann_products)}")
print(f"Cannibalized records : {df['Cannibalization_Flag'].sum():,}")

### B7. Cannibalization Rate

In [ ]:
# Sales lost by P6 (cannibalized peer in G2)
p6_before = df[(df['Product_ID'] == 'P6') & (df['Period_Flag'] == 'Before_Launch')]['Sales'].sum()
p6_after  = df[(df['Product_ID'] == 'P6') & (df['Period_Flag'] == 'After_Launch')]['Sales'].sum()

# Sales gained by new products P4 + P5 (launched in G2)
p4p5_gained = df[
    (df['Product_ID'].isin(['P4', 'P5'])) &
    (df['Period_Flag'] == 'After_Launch')
]['Sales'].sum()

# Normalize for different time periods
months_before = df[df['Period_Flag'] == 'Before_Launch']['Year_Month'].nunique()
months_after  = df[df['Period_Flag'] == 'After_Launch']['Year_Month'].nunique()

p6_loss_norm = (p6_before / months_before) * months_after - p6_after
cann_rate = (p6_loss_norm / p4p5_gained * 100) if p4p5_gained > 0 else 0

print(f"P6 sales loss (normalized) : {p6_loss_norm:,.0f} units")
print(f"P4+P5 sales gained         : {p4p5_gained:,.0f} units")
print(f"Cannibalization Rate       : {cann_rate:.1f}%")

### B8. Marketing Efficiency

In [ ]:
df['Marketing_Efficiency'] = np.where(
    df['Marketing_Spend'] > 0,
    (df['Revenue'] / df['Marketing_Spend']).round(4),
    np.nan
)

print("Top 5 marketing-efficient products:")
print(df.groupby('Product_ID')['Marketing_Efficiency'].mean().round(2).sort_values(ascending=False).head())

### B9. Price Band

In [ ]:
bins   = [0, 50, 100, 200, 500]
labels = ['Budget (0-50)', 'Mid (51-100)', 'Premium (101-200)', 'Luxury (201+)']
df['Price_Band'] = pd.cut(df['Price'], bins=bins, labels=labels)

print("Price Band Distribution:")
print(df['Price_Band'].value_counts().sort_index().to_string())

### B10. Stock Impact & Customer Switching

In [ ]:
# Stock Impact Flag (1 = out of stock)
df['Stock_Impact_Flag'] = np.where(df['Stock_Available'] == 0, 1, 0)
print(f"Out-of-stock records: {df['Stock_Impact_Flag'].sum():,}")

# Customer Switching: bought P6 before, then P4/P5 after
cust_p6_before = set(df[(df['Product_ID'] == 'P6') & (df['Period_Flag'] == 'Before_Launch')]['Customer_ID'])
cust_new_after = set(df[(df['Product_ID'].isin(['P4', 'P5'])) & (df['Period_Flag'] == 'After_Launch')]['Customer_ID'])
switchers = cust_p6_before & cust_new_after

switching_rate = len(switchers) / len(cust_p6_before) * 100 if cust_p6_before else 0
print(f"Customer Switching Rate: {switching_rate:.1f}%")

# Demand Shift Flag
df['Demand_Shift_Flag'] = (
    df['Customer_ID'].isin(switchers) &
    df['Product_ID'].isin(['P4', 'P5']) &
    (df['Period_Flag'] == 'After_Launch')
).astype(int)
print(f"Demand shift records: {df['Demand_Shift_Flag'].sum():,}")

---
## Part C: Save Cleaned Dataset

In [ ]:
output_path = "../Data/cannibalization_cleaned.csv"
df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

print(f"\nNew columns added:")
new_cols = ['Period_Flag', 'Is_Launched_Product', 'Is_Affected_Group',
            'Revenue_Contribution_Pct', 'Year', 'Month', 'Month_Name',
            'Quarter', 'Year_Month', 'Sales_Change_Pct', 'Cannibalization_Flag',
            'Marketing_Efficiency', 'Price_Band', 'Stock_Impact_Flag', 'Demand_Shift_Flag']
for c in new_cols:
    print(f"  + {c}")

print("\nPhase 2 Complete. Proceed to Phase 3.")